# T4 — Positional deep baseline: the one-hot nucleotide CNN

**Question answered.** Every encoding used elsewhere in the paper (raw/TF-IDF k-mer, Word2Vec,
CGR) destroys sequence order before the network sees it. This notebook adds a position-preserving
one-hot nucleotide CNN (the encoding Fabijanska & Grabowski / VGDC used, Table 6) and evaluates it
at every fragment length, so the paper's claim becomes "even a positionally-aware CNN does not
beat a linear k-mer model" (if that holds) instead of an unsupported architectural claim.

**Be ready for either outcome.** The one-hot CNN may beat the linear SVM at short lengths,
because positional motifs can survive fragmentation better than k-mer composition estimates.
That is a valid, reportable result and changes the paper's framing -- this notebook does not
assume which way it goes.

**GPU needed.** Yes. Turn on a GPU T4 x2 accelerator.

**Runtime.** 7 lengths (300, 400, 500, 750, 1000, 1500, full) x up to 5 seeds. Each fit is a
few minutes with early stopping. At 5 seeds x 7 lengths = 35 fits x ~3 min &asymp; 1.5-2 h.
`FAST_MODE` below drops to 3 seeds and 4 lengths if you need to fit this alongside other GPU
notebooks in your weekly quota.

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


### 1. Imports

In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model, Input

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, balanced_accuracy_score, matthews_corrcoef, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC


### 2. Config

In [ ]:

FAST_MODE = False
SEEDS   = [0, 1, 2] if FAST_MODE else [0, 1, 2, 3, 4]
LENGTHS = [300, 500, 1000, None] if FAST_MODE else [300, 400, 500, 750, 1000, 1500, None]  # None = full length
MAXLEN  = 3500          # covers the dataset's length range (1,746-3,504 nt)
EPOCHS  = 80
BATCH   = 48
OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)


### 3. Load data

In [ ]:

DATA_PATH = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)
le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all = df["sequence"].values
grp_all = df["cluster"].values
print(f"{len(df)} sequences | {n_cls} classes")


### 4. Split, fragment, one-hot encode

In [ ]:

def grouped_split(X, y, grp, seed):
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(X[trv], y[trv], grp[trv]))
    return trv[tr_rel], trv[va_rel], te

def fragment(seq, length, rng):
    if length is None or len(seq) <= length:
        return seq
    start = rng.integers(0, len(seq) - length + 1)
    return seq[start:start+length]

MAP = {"A": 0, "C": 1, "G": 2, "T": 3}

def onehot(seq, maxlen=MAXLEN):
    """Position-preserving encoding, shape (maxlen, 4). Row i = one-hot of base i.
    N / ambiguity codes -> all-zero row (a genuine 'no information' signal the conv
    filters can learn to skip). Post-padded with zeros."""
    x = np.zeros((maxlen, 4), dtype=np.float32)
    for i, b in enumerate(seq[:maxlen]):
        j = MAP.get(b)
        if j is not None:
            x[i, j] = 1.0
    return x

def build_onehot_batch(seqs, length, rng, maxlen=MAXLEN):
    frags = [fragment(s, length, rng) for s in seqs]
    return np.stack([onehot(s, maxlen) for s in frags])


### 5. The positional CNN (wide first-layer motif filters, global pooling -> length-tolerant)

In [ ]:

def build_seqcnn(maxlen=MAXLEN, n_classes=10):
    inp = Input(shape=(maxlen, 4))
    x = inp
    for filters, width in [(64, 15), (128, 9), (128, 9), (256, 5)]:
        x = layers.Conv1D(filters, width, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling1D(4, padding="same")(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return Model(inp, out, name="seqcnn_onehot")

def macro_f1_metric_fn(y_true, y_pred_probs):
    y_pred = np.argmax(y_pred_probs, axis=1)
    return f1_score(y_true, y_pred, average="macro")

class MacroF1EarlyStop(tf.keras.callbacks.Callback):
    """Early-stop on validation macro-F1, not loss -- loss is majority-class-dominated
    on a 10:1 imbalanced problem (per reviewer note T4/T7)."""
    def __init__(self, X_val, y_val, patience=10):
        super().__init__()
        self.X_val, self.y_val, self.patience = X_val, y_val, patience
        self.best = -1; self.wait = 0; self.best_weights = None
    def on_epoch_end(self, epoch, logs=None):
        probs = self.model.predict(self.X_val, verbose=0)
        f1 = macro_f1_metric_fn(self.y_val, probs)
        if f1 > self.best:
            self.best = f1; self.wait = 0; self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
                self.model.set_weights(self.best_weights)


### 6. Main loop: every length, every seed

In [ ]:

rows = []
for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tr, va, te = grouped_split(X_all, y_all, grp_all, seed)
    ytr, yva, yte = y_all[tr], y_all[va], y_all[te]
    rng = np.random.default_rng(seed)

    for L in LENGTHS:
        Xtr = build_onehot_batch(X_all[tr], L, rng)
        Xva = build_onehot_batch(X_all[va], L, rng)
        Xte = build_onehot_batch(X_all[te], L, rng)

        tf.keras.backend.clear_session()
        model = build_seqcnn(MAXLEN, n_cls)
        cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))
        model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                       loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                       metrics=["accuracy"])
        cb = MacroF1EarlyStop(Xva, yva, patience=10)
        model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS, batch_size=BATCH,
                  class_weight=cw, callbacks=[cb], verbose=0)

        probs = model.predict(Xte, verbose=0)
        pred = np.argmax(probs, axis=1)
        row = dict(model="onehot_cnn", length=(L if L else "full"),
                   length_num=(L if L else 9999), seed=seed,
                   f1=f1_score(yte, pred, average="macro"),
                   bal_acc=balanced_accuracy_score(yte, pred),
                   mcc=matthews_corrcoef(yte, pred),
                   acc=accuracy_score(yte, pred))
        rows.append(row)
        print(row)
        pd.DataFrame(rows).to_csv(f"{OUT}/t4_onehot_cnn_raw.csv", index=False)  # incremental save


### 7. Compare against the linear SVM baseline at the same lengths (for the crossover plot)

In [ ]:

def fragment_all(seqs, length, rng):
    return [fragment(s, length, rng) for s in seqs]

svm_rows = []
for seed in SEEDS:
    tr, va, te = grouped_split(X_all, y_all, grp_all, seed)
    ytr, yte = y_all[tr], y_all[te]
    rng = np.random.default_rng(seed)
    for L in LENGTHS:
        seq_tr = fragment_all(X_all[tr], L, rng)
        seq_te = fragment_all(X_all[te], L, rng)
        vec = TfidfVectorizer(analyzer="char", ngram_range=(6,6), max_features=512, lowercase=False)
        Xtr = vec.fit_transform(seq_tr); Xte = vec.transform(seq_te)
        clf = LinearSVC(class_weight="balanced", random_state=seed, max_iter=20000)
        clf.fit(Xtr, ytr)
        pred = clf.predict(Xte)
        svm_rows.append(dict(model="linear_svm_tfidf6", length=(L if L else "full"),
                              length_num=(L if L else 9999), seed=seed,
                              f1=f1_score(yte, pred, average="macro"),
                              bal_acc=balanced_accuracy_score(yte, pred)))
pd.DataFrame(svm_rows).to_csv(f"{OUT}/t4_svm_comparator_raw.csv", index=False)


### 8. Aggregate + crossover plot

In [ ]:

raw_cnn = pd.DataFrame(rows)
raw_svm = pd.DataFrame(svm_rows)
raw_all = pd.concat([raw_cnn, raw_svm], ignore_index=True)
raw_all.to_csv(f"{OUT}/t4_combined_raw.csv", index=False)

summ = raw_all.groupby(["model","length","length_num"])[["f1","bal_acc"]].agg(["mean","std"]).round(4)
summ.columns = [f"{a}_{b}" for a,b in summ.columns]
summ = summ.reset_index().sort_values(["model","length_num"])
summ.to_csv(f"{OUT}/t4_summary.csv", index=False)
print(summ)

fig, ax = plt.subplots(figsize=(7,5))
for m, g in summ.groupby("model"):
    g = g.sort_values("length_num")
    ax.errorbar(g["length_num"], g["f1_mean"], yerr=g["f1_std"], marker="o", label=m)
ax.set_xlabel("fragment length (bp); 9999 = full length")
ax.set_ylabel("macro-F1")
ax.set_title("T4: one-hot positional CNN vs. linear k-mer SVM")
ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT}/t4_crossover_plot.png", dpi=200)
plt.show()


### 9. Zip outputs

In [ ]:
!cd /kaggle/working && zip -r t4_onehot_cnn_outputs.zip t4_onehot_cnn_raw.csv t4_svm_comparator_raw.csv t4_combined_raw.csv t4_summary.csv t4_crossover_plot.png